# 04 · Provocar y diagnosticar sobreajuste


Ejecuta las celdas en orden. Antes de mirar la solución, intenta la actividad y anota una predicción. Si cambias parámetros, vuelve a ejecutar desde la celda de creación del modelo: llamar a `fit` otra vez continúa el entrenamiento anterior.

**Entorno:** Google Colab o Jupyter con `pip install tensorflow numpy matplotlib scikit-learn notebook`. Reinicia el kernel tras instalar si es necesario. No se necesitan descargas de conjuntos de datos. Los resultados y tiempos pueden variar por versión, CPU y aleatoriedad.

**Tiempo:** 40–45 minutos.

## 1. Hipótesis del experimento
Entrenaremos una red con mucha capacidad sobre solo 200 imágenes de entrenamiento. Para hacer más visible la memorización, alteraremos aleatoriamente el 35 % de las etiquetas **solo en entrenamiento**. Las etiquetas de validación permanecen correctas. El propósito es educativo: el ruido es artificial y genera una diferencia adicional entre entrenamiento y validación, así que la brecha no representa por sí sola el sobreajuste con etiquetas limpias.

Una señal será que la pérdida de entrenamiento siga bajando mientras la de validación sube después de un mínimo. La exactitud de validación puede estabilizarse mientras la pérdida sube: la pérdida también refleja la confianza. El resultado concreto depende del entorno.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

In [ ]:
def curvas(h, metrica="loss"):
    plt.figure(figsize=(7, 4))
    plt.plot(h.history[metrica], label="Entrenamiento")
    plt.plot(h.history["val_" + metrica], label="Validación")
    plt.xlabel("Época")
    plt.ylabel(metrica)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

imagenes, etiquetas = load_digits(return_X_y=True)
# Imágenes reales de 8x8 en escala de grises, etiquetas de 0 a 9.
X = (imagenes.reshape(-1, 8, 8, 1) / 16.0).astype("float32")
y = etiquetas.astype("int32")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print("Entrenamiento, validación, prueba:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# Una muestra estratificada fija; se conserva el mismo conjunto de validación.
X_peq, _, y_limpia, _ = train_test_split(
    X_train, y_train, train_size=200, stratify=y_train, random_state=42
)
rng = np.random.default_rng(42)
y_ruidosa = y_limpia.copy()
indices = rng.choice(len(y_ruidosa), size=int(0.35 * len(y_ruidosa)), replace=False)
# Cambia cada etiqueta elegida por una clase distinta, sin tocar validación.
y_ruidosa[indices] = (y_ruidosa[indices] + rng.integers(1, 10, len(indices))) % 10
print(
    "Entrenamiento:",
    len(X_peq),
    "| Etiquetas alteradas:",
    np.sum(y_ruidosa != y_limpia),
    "| Validación limpia:",
    len(y_val),
)

## 2. Provoca la memorización

Predice la forma de las curvas. Entrenamos durante 100 épocas, sin parada temprana, para observar qué pasa después del mejor punto de validación. Puede tardar unos minutos en CPU.

In [ ]:
def red_grande():
    keras.utils.set_random_seed(42)
    modelo = keras.Sequential(
        [
            keras.Input(shape=(8, 8, 1)),
            keras.layers.Flatten(),
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    modelo.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


modelo = red_grande()
modelo.summary()
h = modelo.fit(
    X_peq,
    y_ruidosa,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    verbose=0,
)
curvas(h)
curvas(h, "accuracy")
mejor = int(np.argmin(h.history["val_loss"]))
print("Mejor época de validación:", mejor + 1)
print(
    "Pérdida val mínima / final:",
    round(h.history["val_loss"][mejor], 4),
    "/",
    round(h.history["val_loss"][-1], 4),
)
print(
    "Exactitud train sobre etiquetas alteradas al final:",
    round(h.history["accuracy"][-1], 4),
)

La red de este experimento es densa para acelerar la memorización; la CNN se estudia en el notebook 03. El mínimo de validación identifica una época candidata, pero no mide desempeño final independiente.

## 3. Tu turno: mitiga y explica

Añade `EarlyStopping` que supervise `val_loss`, con paciencia 8 y `restore_best_weights=True`. Entrena un modelo nuevo sobre los mismos datos y compara el mínimo, el final de las curvas y la época detenida. ¿Qué puede y qué no puede corregir la parada temprana?

In [ ]:
# Tu código aquí.

**Mi interpretación:** escribe aquí.

## 4. Solución de referencia

In [ ]:
modelo_es = red_grande()
parada = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True
)
h_es = modelo_es.fit(
    X_peq,
    y_ruidosa,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[parada],
    verbose=0,
)
curvas(h_es)
print("Épocas ejecutadas:", len(h_es.history["loss"]))
print("Mejor época:", np.argmin(h_es.history["val_loss"]) + 1)
print(
    "Pérdida val con pesos restaurados:",
    modelo_es.evaluate(X_val, y_val, verbose=0, return_dict=True)["loss"],
)

La parada temprana evita seguir entrenando cuando no mejora la pérdida de validación durante la paciencia y restaura los pesos de la mejor época observada. No repara las etiquetas incorrectas ni garantiza buena generalización. El último valor de `history` corresponde a la última época ejecutada, no necesariamente a los pesos restaurados.

## 5. Evaluación y límites

Si el diseño ya quedó fijado usando validación, evalúa una sola vez el modelo elegido sobre prueba. En un proyecto real, investiga la calidad de etiquetas y distribuciones, y si es posible obtén más datos representativos.

In [ ]:
print(
    "Prueba del modelo con parada temprana:",
    modelo_es.evaluate(X_test, y_test, verbose=0, return_dict=True),
)

## 6. Desafío opcional: aísla el ruido

Entrena desde cero la misma red con `y_limpia` en lugar de `y_ruidosa` y compara curvas de validación. Mantén tamaño, semillas y épocas. ¿Persiste la brecha? La diferencia entre ambos experimentos ayuda a separar el efecto del ruido, aunque la variabilidad de entrenamiento aún puede influir. Usa validación, no prueba, para esta comparación.